# Calculate daily China AQI labels and prepare model splits
This notebook loads hourly readings from the `all_regions` table in the MySQL `air_quality` database. It calculates one daily AQI and classification per station and calendar day, then creates reproducible stratified training and test sets. MySQL is required to load the source table; database writes for the split tables remain optional.


## China AQI basis and six-level classification

This notebook uses China’s AQI breakpoints, not WHO guideline values. The Ministry response linked below explains the HJ 633-2012 rationale; the current technical regulation is HJ 633-2026.

The output has one row per station and calendar day. PM2.5, PM10, SO2, NO2, and CO use 24-hour daily means, requiring at least 20 valid hourly readings for each pollutant. O3 uses the day’s maximum trailing 8-hour average; each window requires 6 valid readings and a day requires at least 14 valid windows ending from 08:00 through 23:00. When the daily maximum 8-hour O3 concentration exceeds 800 µg/m³, the daily maximum 1-hour O3 concentration is used for its IAQI. CO source values are converted from µg/m³ to mg/m³.

Each pollutant IAQI is linearly interpolated from its concentration breakpoints. The overall daily AQI is the maximum available pollutant IAQI.

| Category code | AQI range | China category |
| ---: | ---: | --- |
| 1 | 0–50 | Excellent |
| 2 | 51–100 | Good |
| 3 | 101–150 | Lightly polluted |
| 4 | 151–200 | Moderately polluted |
| 5 | 201–300 | Heavily polluted |
| 6 | 301–500 | Severely polluted |

| AQI breakpoint | PM2.5 | PM10 | SO2 (24h) | NO2 (24h) | CO (24h) | O3 (8h) |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 50 | 30 | 50 | 50 | 40 | 2 | 100 |
| 100 | 60 | 120 | 150 | 80 | 4 | 160 |
| 150 | 115 | 250 | 475 | 180 | 14 | 215 |
| 200 | 150 | 350 | 800 | 280 | 24 | 265 |
| 300 | 250 | 420 | 1600 | 565 | 36 | 800 |
| 400 | 350 | 500 | 2100 | 750 | 48 | — |
| 500 | 500 | 600 | 2620 | 940 | 60 | — |

Concentration units are µg/m³ except CO, which is mg/m³. The input must contain no duplicate station-hour records. Sources: [Ministry of Ecology and Environment, linked response](https://www.mee.gov.cn/xxgk2018/xxgk/xxgk13/201810/t20181015_662213_wh.html), [HJ 633-2026 standard page](https://www.mee.gov.cn/ywgz/fgbz/bz/bzwb/jcffbz/202602/t20260225_1144441.shtml).

In [1]:
import os
import re
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

# The China AQI rules and the dataset split live in aqi_labels.py beside this notebook,
# so this notebook and model_development.ipynb share one implementation instead of two
# copies that drift apart.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aqi_labels import (  # noqa: E402  (import must follow the path bootstrap)
    AQI_CATEGORIES,
    DAILY_POLLUTANTS,
    POLLUTANTS,
    POLLUTANT_BREAKPOINTS,
    assert_disjoint_station_days,
    build_daily_labels,
    split_by_station_day,
)

# Uses this local default if AIR_QUALITY_DB_URL is not set.
# Set AIR_QUALITY_DB_URL to override it when your MySQL credentials differ.
DB_URL = os.environ.get(
    "AIR_QUALITY_DB_URL",
    "mysql+pymysql://root:@localhost/air_quality",
)
SOURCE_TABLE = "all_regions"
WRITE_SPLITS_TO_DB = False
TEST_FRACTION = 0.20
RANDOM_STATE = 42

if not re.fullmatch(r"[A-Za-z0-9_]+", SOURCE_TABLE):
    raise ValueError(f"Invalid MySQL table name: {SOURCE_TABLE!r}")

engine = create_engine(DB_URL, pool_pre_ping=True)
query = text(f"SELECT * FROM {SOURCE_TABLE}")

try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(query, connection)
except Exception as error:
    raise RuntimeError(
        f"Could not load table {SOURCE_TABLE!r} from the air_quality database. "
        "Check that MySQL is running, the database and table exist, "
        "AIR_QUALITY_DB_URL has the correct credentials, and PyMySQL is installed. "
        f"Original database error: {error}"
    ) from error

print(f"Loaded {len(source_df):,} rows from air_quality.{SOURCE_TABLE}")
display(source_df.head())

ModuleNotFoundError: No module named 'aqi_labels'

## Calculate daily averages and China AQI values

Generate one result per station and calendar day. Daily means for PM2.5, PM10, SO2, NO2, and CO require at least 20 valid hourly readings. For O3, use the day’s maximum valid trailing 8-hour average (at least 6 readings per window and at least 14 valid windows ending from 08:00 through 23:00). Convert daily concentrations to pollutant IAQIs, take the maximum available IAQI as the daily AQI, and assign its China category.

In [ ]:
# The China AQI calculation now lives in aqi_labels.py, imported by the load cell above.
# This cell keeps only the reporting that belongs to this notebook.
daily_aqi_df, hourly_labeled_df = build_daily_labels(source_df)
labeled_df = daily_aqi_df

print("Daily pollutant IAQI counts:")
for pollutant in POLLUTANTS:
    print(f"{pollutant}:")
    display(
        labeled_df[f"{pollutant}_iaqi"]
        .value_counts(dropna=True)
        .sort_index()
        .rename("station-days")
    )
print("\nDaily China AQI category counts:")
display(
    labeled_df["aqi_category"]
    .value_counts()
    .reindex(AQI_CATEGORIES.values(), fill_value=0)
    .rename("station-days")
)
print(f"Daily station records: {len(labeled_df):,}")

labeled_df = hourly_labeled_df
print(f"Hourly rows with daily AQI labels: {labeled_df['aqi'].notna().sum():,} / {len(labeled_df):,}")

## Interpolation provenance and measured-only sensitivity

`etl.ipynb` filled short gaps in the pollutant series by linear interpolation (at most 3 consecutive hours, within each station) and logged the columns it touched in `interpolated_columns`. Downstream those filled hours are indistinguishable from measured readings, so they **count toward the validity gates above**.

The labels are not changed here. This section measures how much of them rests on interpolation, then re-runs the same calculation under a stricter rule:

| Rule | Hours allowed to satisfy a gate |
| --- | --- |
| **Primary** (used downstream) | measured **or** interpolated |
| **Measured only** (sensitivity check) | measured only |

The deciding number is the **marginal station-day** count — days that clear a gate *only* because interpolated hours count. It is assessed per pollutant, since a missing CO hour does not invalidate that day's PM2.5 average. Negligible means the distinction does not affect the analysis; large means the limitation must be stated alongside the labels.

**Known limitation:** `RAIN` was zero-filled rather than interpolated, and those 390 fills are absent from `interpolated_columns`. The measured-only rule is therefore exact for the six pollutants and `wd`, and approximate for `RAIN`.

In [ ]:
# Option C sensitivity: rebuild the daily labels from measured hours only.
# The primary labels are left untouched; this only measures how much they depend on interpolation.
sensitivity_daily_df, sensitivity_hourly_df = build_daily_labels(source_df, exclude_interpolated=True)

print("Interpolation behind the primary labels (hours that were filled rather than measured):")
provenance_rows = []
for pollutant in POLLUTANT_BREAKPOINTS:
    interpolated_hours = int(daily_aqi_df[f"{pollutant}_interpolated_hours"].sum())
    station_days_with_fills = int((daily_aqi_df[f"{pollutant}_interpolated_hours"] > 0).sum())
    row = {
        "pollutant": pollutant,
        "interpolated_hours": interpolated_hours,
        "station_days_with_fills": station_days_with_fills,
        "share_of_station_days": station_days_with_fills / len(daily_aqi_df),
        "interpolated_share_of_valid_hours": float("nan"),
        # Ozone uses a rolling-window rule, so its marginality comes from the rerun below.
        "marginal_station_days": float("nan"),
    }
    if pollutant in DAILY_POLLUTANTS:
        valid_hours = int(daily_aqi_df[f"{pollutant}_valid_hours"].sum())
        if valid_hours:
            row["interpolated_share_of_valid_hours"] = interpolated_hours / valid_hours
        row["marginal_station_days"] = int(daily_aqi_df[f"{pollutant}_marginal_pass"].sum())
    provenance_rows.append(row)
display(pd.DataFrame(provenance_rows).set_index("pollutant"))

marginal_days = int(
    daily_aqi_df[[f"{pollutant}_marginal_pass" for pollutant in DAILY_POLLUTANTS]].any(axis=1).sum()
)
print(
    f"Station-days clearing the 20-hour gate only because interpolated hours count: "
    f"{marginal_days:,} of {len(daily_aqi_df):,} ({marginal_days / len(daily_aqi_df):.2%})"
)

# Days that get no AQI at all. If their missingness is concentrated in one station
# it biases the surviving sample rather than thinning it at random.
unlabeled_days = daily_aqi_df.loc[daily_aqi_df["aqi"].isna()]
print(f"\nStation-days with no AQI under the primary rule: {len(unlabeled_days):,}")
if unlabeled_days.empty:
    print("Every station-day received an AQI.")
else:
    display(unlabeled_days["station"].value_counts().rename("station-days").to_frame())

# Compare the two rules station-day by station-day.
comparison = daily_aqi_df[["station", "date", "aqi", "aqi_category"]].merge(
    sensitivity_daily_df[["station", "date", "aqi", "aqi_category"]],
    on=["station", "date"],
    how="outer",
    suffixes=("_primary", "_measured"),
    validate="one_to_one",
)
primary_has = comparison["aqi_primary"].notna()
measured_has = comparison["aqi_measured"].notna()
both_has = primary_has & measured_has

print("\nLabels under each rule:")
display(pd.DataFrame(
    {
        "primary": [int(primary_has.sum()), int(hourly_labeled_df["aqi"].notna().sum())],
        "measured_only": [int(measured_has.sum()), int(sensitivity_hourly_df["aqi"].notna().sum())],
    },
    index=["station-days with an AQI", "hourly rows with an AQI label"],
))
print(f"Station-days that lose their AQI entirely: {int((primary_has & ~measured_has).sum()):,}")
print(f"Station-days that gain an AQI: {int((~primary_has & measured_has).sum()):,} (expected 0)")

category_order = list(AQI_CATEGORIES.values())
category_comparison = pd.DataFrame({
    "primary": daily_aqi_df["aqi_category"].value_counts().reindex(category_order, fill_value=0),
    "measured_only": sensitivity_daily_df["aqi_category"].value_counts().reindex(category_order, fill_value=0),
})
category_comparison["difference"] = category_comparison["measured_only"] - category_comparison["primary"]
category_comparison["percent_change"] = (
    category_comparison["difference"]
    / category_comparison["primary"].where(category_comparison["primary"] > 0) * 100
).round(2)
print("\nDaily China AQI category counts under each rule:")
display(category_comparison)

# Of the station-days both rules can label, how many change category, and which way?
changed = comparison.loc[both_has & comparison["aqi_category_primary"].ne(comparison["aqi_category_measured"])]
print(
    f"\nStation-days labelled by both rules whose category changes: {len(changed):,} "
    f"of {int(both_has.sum()):,} ({len(changed) / max(int(both_has.sum()), 1):.2%})"
)
if changed.empty:
    print("No shared station-day changes category, so the distinction does not move the labels.")
else:
    display(pd.crosstab(
        changed["aqi_category_primary"], changed["aqi_category_measured"]
    ).reindex(index=category_order, columns=category_order, fill_value=0))

    severity = {name: code for code, name in AQI_CATEGORIES.items()}
    shift = changed["aqi_category_measured"].map(severity) - changed["aqi_category_primary"].map(severity)
    print("Severity shift among changed days (positive = the stricter rule rates the day worse):")
    display(shift.value_counts().sort_index().rename("station-days").to_frame())

## Create the training and test sets

An 80/20 split is taken separately within each station of the complete cleaned source table, so every station appears in both sets and keeps roughly its share of rows.

The sampling unit is the **station-day**, not the hourly row: all hours of a given station and date go to the same side. This matters because the models predict a *daily* category from *hourly* rows — under an hourly-row split, hours of one station-day could land in both sets and test rows would be scored against a label their own training rows had already seen. The split keeps all source columns and rows and does not use the daily AQI table. The sets remain in memory as `train_set` and `test_set`.

In [ ]:
# Group-aware 80/20 split from aqi_labels.py. The sampling unit is the station-day, so
# all 24 hours of a station-day land on the same side and no daily label is shared across
# the boundary. Sampling still happens within each station, so every station appears in
# both sets and keeps roughly its share of rows.
train_set, test_set = split_by_station_day(
    source_df, test_fraction=TEST_FRACTION, random_state=RANDOM_STATE
)

if len(train_set) + len(test_set) != len(source_df):
    raise RuntimeError("The split did not partition the source rows.")
train_days, test_days = assert_disjoint_station_days(train_set, test_set)

print(f"Complete source rows: {len(source_df):,}")
print(f"Training set: {len(train_set):,} rows")
print(f"Test set: {len(test_set):,} rows")
print(f"Total after split: {len(train_set) + len(test_set):,} rows")
print(f"Station-days: {train_days:,} training / {test_days:,} test, no station-day in both")

# Saving these tables to MySQL is optional and is done in the final cell of this notebook.

## Optionally save the split tables in MySQL

The default is `False` to avoid changing the database unexpectedly. Set `WRITE_SPLITS_TO_DB = True` after reviewing the generated counts.

The tables contain only the original hourly columns, because the split is taken from `source_df` in the cell above. The matching station-day pollutant averages, IAQI values, daily AQI, and both the overall and per-pollutant daily categories are calculated earlier into `labeled_df` and are **not** written by this cell. Pass `labeled_df` to `split_by_station_day` instead if the saved tables should also carry the daily labels.

Use versioned table names instead if multiple experiments must be retained.

In [ ]:
if WRITE_SPLITS_TO_DB:
    DB_URL = os.environ.get("AIR_QUALITY_DB_URL")
    if not DB_URL:
        raise RuntimeError("Set AIR_QUALITY_DB_URL before enabling database writes.")
    from sqlalchemy import create_engine
    engine = create_engine(DB_URL)
    train_set.to_sql("training_set", engine, if_exists="replace", index=False)
    test_set.to_sql("test_set", engine, if_exists="replace", index=False)
    print("Saved training_set and test_set tables to air_quality")
else:
    print("Database writes disabled. Set WRITE_SPLITS_TO_DB = True to save the split tables.")


Database writes disabled. Set WRITE_SPLITS_TO_DB = True to save the split tables.
